# ETAPA: Quadro de Qualidade de Dados (DAMA) por Forno e Dominio - Entregavel 1A

Objetivo: gerar indicadores consolidado por forno e por dominio (completude, validade, acuracia-proxy, tempestividade, unicidade, consistencia), com interpretacao operacional e acoes recomendadas.

Regras: somente leitura, saidas em outputs/R3Q, sem emojis.


In [1]:
from __future__ import annotations

import json
import csv
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 220)

BASE_DIR = Path('/home/wilson/Maringa/fase_1_diagnostico')
OUT_DIR = Path('/home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R3Q')
OUT_DIR.mkdir(parents=True, exist_ok=True)

P_FILE_INDEX = Path('/home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R2/r2_file_index.csv')
P_PARSE_AUDIT = Path('/home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R2/r2_parse_audit.csv')

for p in [P_FILE_INDEX, P_PARSE_AUDIT]:
    if not p.exists():
        raise FileNotFoundError(str(p))

RUN_TS = datetime.now().strftime('%Y-%m-%d_%H%M%S')
print('RUN_TS:', RUN_TS)
print('OUT_DIR:', OUT_DIR)



RUN_TS: 2026-01-06_083804
OUT_DIR: /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R3Q


In [2]:
df_idx = pd.read_csv(P_FILE_INDEX)
df_audit = pd.read_csv(P_PARSE_AUDIT)

print('r2_file_index (head 20):')
display(df_idx.head(20))
print('r2_parse_audit (head 20):')
display(df_audit.head(20))

# Validar colunas minimas esperadas
required_cols = {'dominio'}
if not required_cols.issubset(set(df_idx.columns)):
    raise RuntimeError(f'Esperado r2_file_index conter colunas {required_cols}. Encontrado: {list(df_idx.columns)}')

# Garantir coluna forno (pode nao existir no r2_file_index)
import re

def extract_forno(rel: str) -> str:
    if not isinstance(rel, str):
        return ''
    m = re.search(r'\bF[1-9]\b', rel)
    if m:
        return m.group(0)
    m2 = re.search(r'(F[1-9])_', rel)
    if m2:
        return m2.group(1)
    return ''

if 'forno' not in df_idx.columns:
    rel_series = df_idx['rel_norm'] if 'rel_norm' in df_idx.columns else df_idx[df_idx.columns[0]].astype(str)
    df_idx['forno'] = [extract_forno(r) for r in rel_series.astype(str)]

# Descobrir coluna de path automaticamente
path_candidates = [c for c in df_idx.columns if 'path' in c.lower()]
if len(path_candidates) == 0:
    raise RuntimeError('Nao encontrei coluna de path no r2_file_index.csv')

path_col = 'path_local' if 'path_local' in df_idx.columns else path_candidates[0]
print('Usando coluna de caminho:', path_col)



r2_file_index (head 20):


,path_local,rel_norm,dominio,suffix,size_bytes,in_raw_dir
0,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2019_F1_Consumo.csv,Consumo Fornos,.csv,42566405,True
1,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2020_F1_Consumo.csv,Consumo Fornos,.csv,28785370,True
2,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2021_F1_Consumo.csv,Consumo Fornos,.csv,29619879,True
3,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2022_F1_Consumo.csv,Consumo Fornos,.csv,29118391,True
4,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2023_F1_Consumo.csv,Consumo Fornos,.csv,28111195,True
5,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2024_F1_Consumo.csv,Consumo Fornos,.csv,27148206,True
6,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2025_F1_Consumo.csv,Consumo Fornos,.csv,25851580,True
7,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2018_F2_Consumo.csv,Consumo Fornos,.csv,28950526,True
8,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2019_F2_Consumo.csv,Consumo Fornos,.csv,28760963,True
9,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2020_F2_Consumo.csv,Consumo Fornos,.csv,29180244,True


r2_parse_audit (head 20):


,path_local,rel_norm,dominio,suffix,ok,encoding_used,delimiter,n_rows,n_cols,columns,date_cols_guess,min_date,max_date,error
0,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2019_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",961515,42,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2019-01-01 00:00:00+00:00,2019-12-31 00:00:00+00:00,NaN
1,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2020_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",1019752,26,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2020-01-01 00:00:00+00:00,2020-12-31 00:00:00+00:00,NaN
2,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2021_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",1012132,27,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2021-01-01 00:00:00+00:00,2021-12-31 00:00:00+00:00,NaN
3,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2022_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",989761,27,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2022-01-01 00:00:00+00:00,2022-12-31 00:00:00+00:00,NaN
4,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2023_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",959436,27,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2023-01-01 00:00:00+00:00,2023-12-31 00:00:00+00:00,NaN
5,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2024_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",925925,27,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2024-01-01 00:00:00+00:00,2024-12-31 00:00:00+00:00,NaN
6,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2025_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",907889,27,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2025-01-01 00:00:00+00:00,2025-04-28 00:00:00+00:00,NaN
7,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2018_F2_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",1019780,26,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2018-01-01 00:00:00+00:00,2018-12-31 00:00:00+00:00,NaN
8,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2019_F2_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",1019762,26,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2019-01-01 00:00:00+00:00,2019-12-31 00:00:00+00:00,NaN
9,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2020_F2_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",997916,27,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2020-01-01 00:00:00+00:00,2020-12-31 00:00:00+00:00,NaN


Usando coluna de caminho: path_local


In [3]:
def detect_time_col(columns: list[str]) -> str | None:
    cands = []
    for c in columns:
        cl = c.strip().lower()
        if cl in {'data', 'date', 'datetime', 'timestamp', 'dt', 'hora', 'datahora', 'data_hora'}:
            cands.append(c)
        elif 'timestamp' in cl or 'date' in cl or 'data' in cl:
            cands.append(c)
    return cands[0] if len(cands) > 0 else None


def robust_outlier_rate(x: pd.Series) -> float:
    v = pd.to_numeric(x, errors='coerce')
    v = v[np.isfinite(v)]
    if v.size < 50:
        return float('nan')
    med = np.median(v)
    mad = np.median(np.abs(v - med))
    if mad == 0:
        return 0.0
    z = 0.6745 * (v - med) / mad
    return float(np.mean(np.abs(z) > 6))


def sniff_delimiter(sample_text: str) -> str:
    candidates = [',', ';', '\t', '|']
    counts = {c: sample_text.count(c) for c in candidates}
    return max(counts, key=counts.get)



In [4]:
# Mapear informacoes de parsing do R2 para reuso (delimiter, encoding)
df_audit['path_resolved'] = df_audit['path_local'].apply(lambda x: str(Path(x).resolve()))
audit_map = df_audit.set_index('path_resolved')[['delimiter','encoding_used','dominio']].to_dict('index')

encodings_try = ['utf-8', 'latin-1', 'cp1252']


def scan_file_metrics_csv(path: Path, dominio: str, sep_hint: str | None, enc_hint: str | None, chunksize: int = 200_000, max_rows: int | None = None) -> dict:
    total_rows = 0
    total_cells = 0
    total_missing = 0
    numeric_parse_ok = 0
    numeric_parse_total = 0
    outlier_rates = []
    time_col = None
    time_parse_ok = 0
    time_parse_total = 0
    n_cols_last = 0

    seen_hash = set()
    dup_count = 0
    hash_total = 0

    last_err = None

    enc_list = [enc_hint] if enc_hint else []
    for e in encodings_try:
        if e not in enc_list:
            enc_list.append(e)

    used_sep = sep_hint
    used_enc = None

    for enc in enc_list:
        try:
            if used_sep is None:
                sample = path.read_text(encoding=enc, errors='replace')[:5000]
                used_sep = sniff_delimiter(sample)
            reader = pd.read_csv(
                path,
                sep=used_sep,
                engine='python',
                encoding=enc,
                encoding_errors='replace',
                chunksize=chunksize,
                quoting=csv.QUOTE_NONE if 'supervisorio forno 4' in dominio.lower() else csv.QUOTE_MINIMAL,
                on_bad_lines='skip'
            )
            used_enc = enc
            break
        except Exception as e:
            last_err = e
            used_sep = sep_hint
            continue

    if used_enc is None:
        raise last_err if last_err else RuntimeError('Falha ao abrir CSV')

    for chunk in reader:
        if time_col is None:
            time_col = detect_time_col(list(chunk.columns))

        n = len(chunk)
        n_cols_last = len(chunk.columns)
        total_rows += n
        total_cells += n * len(chunk.columns)
        total_missing += int(chunk.isna().sum().sum())

        for col in chunk.columns:
            s = chunk[col]
            sample = s.dropna().astype(str).head(200)
            if sample.empty:
                continue
            frac_has_digit = np.mean(sample.str.contains(r'\d', regex=True))
            if s.dtype == object and frac_has_digit < 0.6:
                continue
            v = pd.to_numeric(s, errors='coerce')
            ok = np.isfinite(v).sum()
            tot = s.notna().sum()
            if tot == 0:
                continue
            numeric_parse_ok += int(ok)
            numeric_parse_total += int(tot)

        if time_col is not None and time_col in chunk.columns:
            t = pd.to_datetime(chunk[time_col], errors='coerce', utc=True)
            time_parse_ok += int(t.notna().sum())
            time_parse_total += int(chunk[time_col].notna().sum())

        num_cols = []
        for col in chunk.columns:
            if len(num_cols) >= 6:
                break
            vc = pd.to_numeric(chunk[col], errors='coerce')
            if vc.notna().sum() > 1000 and np.isfinite(vc).sum() > 0.8 * vc.notna().sum():
                num_cols.append(col)
        for col in num_cols:
            r = robust_outlier_rate(chunk[col])
            if np.isfinite(r):
                outlier_rates.append(r)

        sample = chunk.head(500)
        for _, row in sample.iterrows():
            h = hash(tuple(row.astype(str).fillna('NA').tolist()))
            if h in seen_hash:
                dup_count += 1
            else:
                seen_hash.add(h)
            hash_total += 1

        if max_rows is not None and total_rows >= max_rows:
            break

    completude = 1.0 - (total_missing / total_cells) if total_cells > 0 else float('nan')
    validade = (numeric_parse_ok / numeric_parse_total) if numeric_parse_total > 0 else float('nan')
    unicidade = 1.0 - (dup_count / hash_total) if hash_total > 0 else float('nan')

    tempestividade = float('nan')
    if time_col is not None and time_parse_total > 0 and (time_parse_ok / max(1, time_parse_total)) > 0.9:
        df_small = pd.read_csv(path, sep=used_sep, engine='python', encoding=used_enc, encoding_errors='replace', nrows=200_000, quoting=csv.QUOTE_NONE if 'supervisorio forno 4' in dominio.lower() else csv.QUOTE_MINIMAL, on_bad_lines='skip')
        if time_col in df_small.columns:
            t = pd.to_datetime(df_small[time_col], errors='coerce', utc=True).dropna().sort_values()
            if len(t) > 10:
                deltas = t.diff().dropna().dt.total_seconds().values
                if len(deltas) > 0:
                    modal = np.median(deltas)
                    if modal > 0:
                        gap_rate = float(np.mean(deltas > 5 * modal))
                        tempestividade = 1.0 - gap_rate

    acuracia_proxy = float(np.nanmean(outlier_rates)) if len(outlier_rates) > 0 else float('nan')

    return {
        'total_rows_scanned': int(total_rows),
        'n_cols': int(n_cols_last),
        'time_col': time_col,
        'completude': float(completude),
        'validade': float(validade),
        'acuracia_proxy_outlier_rate': float(acuracia_proxy),
        'tempestividade': float(tempestividade),
        'unicidade': float(unicidade),
        'sep_used': used_sep,
        'enc_used': used_enc
    }



In [5]:
records = []

# filtrar apenas CSVs conhecidos
if 'suffix' in df_idx.columns:
    df_idx_csv = df_idx[df_idx['suffix'] == '.csv'].copy()
else:
    df_idx_csv = df_idx.copy()

for (dominio, forno), g in tqdm(df_idx_csv.groupby(['dominio', 'forno']), desc='Grupos dominio/forno'):
    files = g[path_col].dropna().astype(str).tolist()
    files = [Path(p) for p in files]

    group_metrics = []
    for p in tqdm(files, desc=f'{dominio} | {forno}', leave=False):
        if not p.exists():
            continue
        key = str(p.resolve())
        sep_hint = audit_map.get(key, {}).get('delimiter') if key in audit_map else None
        enc_hint = audit_map.get(key, {}).get('encoding_used') if key in audit_map else None
        max_rows = 2_000_000 if isinstance(dominio, str) and 'consumo' in dominio.lower() else None
        m = scan_file_metrics_csv(p, dominio=dominio or '', sep_hint=sep_hint, enc_hint=enc_hint, chunksize=200_000, max_rows=max_rows)
        m['dominio'] = dominio
        m['forno'] = forno
        m['file_path'] = str(p)
        group_metrics.append(m)

    if len(group_metrics) == 0:
        continue

    df_g = pd.DataFrame(group_metrics)

    mode_cols = int(df_g['n_cols'].mode().iloc[0]) if len(df_g['n_cols']) > 0 else 0
    consistencia_schema = float((df_g['n_cols'] == mode_cols).mean()) if mode_cols is not None else float('nan')

    rec = {
        'dominio': dominio,
        'forno': forno,
        'n_files': int(len(df_g)),
        'completude': float(df_g['completude'].mean()),
        'validade': float(df_g['validade'].mean()),
        'acuracia_proxy': float(df_g['acuracia_proxy_outlier_rate'].mean()),
        'tempestividade': float(df_g['tempestividade'].mean()),
        'unicidade': float(df_g['unicidade'].mean()),
        'consistencia': float(consistencia_schema),
        'time_cols_detectadas': '|'.join(sorted({c for c in df_g['time_col'].dropna().astype(str)})) if 'time_col' in df_g.columns else ''
    }
    records.append(rec)

quadro = pd.DataFrame(records)
print('Quadro consolidado (head 20):')
display(quadro.head(20))




Grupos dominio/forno:   0%|          | 0/19 [00:00<?, ?it/s]


Consumo Fornos | F1:   0%|          | 0/7 [00:00<?, ?it/s]


Consumo Fornos | F1:  14%|█▍        | 1/7 [00:03<00:21,  3.62s/it]


Consumo Fornos | F1:  29%|██▊       | 2/7 [00:06<00:15,  3.02s/it]


Consumo Fornos | F1:  43%|████▎     | 3/7 [00:08<00:11,  2.86s/it]

...


Supervisorio Forno 5 | F5: 100%|██████████| 1/1 [00:05<00:00,  5.70s/it]


Grupos dominio/forno: 100%|██████████| 19/19 [03:09<00:00, 13.24s/it]


Grupos dominio/forno: 100%|██████████| 19/19 [03:09<00:00, 10.00s/it]

Quadro consolidado (head 20):


,dominio,forno,n_files,completude,validade,acuracia_proxy,tempestividade,unicidade,consistencia,time_cols_detectadas
0,Consumo Fornos,F1,7,0.008234,0.824367,0.048059,NaN,0.188419,0.714286,Data
1,Consumo Fornos,F2,8,0.008489,0.824302,0.042272,NaN,0.185917,0.625000,Data
2,Consumo Fornos,F3,8,0.005964,0.823455,0.052146,NaN,0.179358,0.500000,Data
3,Consumo Fornos,F4,8,0.011532,0.826249,0.062939,NaN,0.181300,0.625000,Data
4,Consumo Fornos,F5,8,0.012073,0.825726,0.064279,NaN,0.181750,0.750000,Data
5,Corridas,F1,8,0.992644,0.798625,0.000271,NaN,1.000000,1.000000,Data_Base
6,Corridas,F2,8,0.979425,0.797562,0.000111,NaN,1.000000,0.625000,Data_Base
7,Corridas,F3,8,0.995088,0.785075,0.000675,NaN,1.000000,1.000000,Data_Base
8,Corridas,F4,8,0.992663,0.779750,0.000591,NaN,1.000000,1.000000,Data_Base
9,Corridas,F5,8,0.992449,0.779704,0.000566,NaN,1.000000,1.000000,Data_Base


In [6]:
def interpretar(row) -> tuple[str, str]:
    probs = []

    def bad(name, cond):
        if cond:
            probs.append(name)

    bad('completude', row['completude'] < 0.98)
    bad('validade', (not np.isfinite(row['validade'])) or row['validade'] < 0.98)
    bad('unicidade', (not np.isfinite(row['unicidade'])) or row['unicidade'] < 0.995)
    bad('consistencia', row['consistencia'] < 0.95)

    if np.isfinite(row['tempestividade']):
        bad('tempestividade', row['tempestividade'] < 0.95)
    else:
        probs.append('tempestividade_indeterminada')

    if np.isfinite(row['acuracia_proxy']):
        bad('acuracia_proxy_alta', row['acuracia_proxy'] > 0.02)
    else:
        probs.append('acuracia_indeterminada')

    # Avaliacao por tipo de modelagem
    def status(ok: bool, risco: str) -> str:
        return 'apto' if ok else risco

    phys_ok = ('consistencia' not in probs) and ('validade' not in probs) and ('tempestividade' not in probs) and ('tempestividade_indeterminada' not in probs)
    sup_ok = ('completude' not in probs) and ('validade' not in probs) and ('unicidade' not in probs)
    unsup_ok = ('completude' not in probs) and ('validade' not in probs)
    rl_ok = phys_ok and ('unicidade' not in probs)

    phys_txt = f"Physics-based: {status(phys_ok, 'risco por consistencia/validade/tempestividade')}."
    sup_txt = f"Supervisionada: {status(sup_ok, 'risco por completude/validade/unicidade')} ."
    unsup_txt = f"Nao supervisionada: {status(unsup_ok, 'risco por completude/validade')} ."
    rl_txt = f"RL: {status(rl_ok, 'risco por consistencia/tempestividade/unicidade')} ."

    if len(probs) == 0:
        interp = 'Dados adequados para analise e modelagem, com risco operacional baixo no nivel de diagnostico. ' + ' '.join([phys_txt, sup_txt, unsup_txt, rl_txt])
        acoes = 'Manter padrao de extracao e registrar dicionario de dados e metadados de medicao.'
        return interp, acoes

    acoes_map = {
        'completude': 'Investigar lacunas e padronizar preenchimento/campos obrigatorios.',
        'validade': 'Padronizar parse/tipos/unidades e bloquear valores nao parseaveis.',
        'unicidade': 'Definir chave natural e deduplicar exportacoes/reprocessamentos.',
        'consistencia': 'Padronizar schema e versionar mudancas de colunas e significados.',
        'tempestividade': 'Corrigir timestamp (fonte, timezone, formato) e alinhar frequencias.',
        'tempestividade_indeterminada': 'Definir coluna temporal oficial e padrao de timestamp por dominio.',
        'acuracia_proxy_alta': 'Revisar calibracao/drift e checar outliers por regras de engenharia.',
        'acuracia_indeterminada': 'Definir referencias de acuracia (calibracao, checagens cruzadas) para sinais criticos.'
    }

    interp = 'Risco moderado/alto para uso direto em modelagem em funcao de: ' + ', '.join(probs) + '. ' + ' '.join([phys_txt, sup_txt, unsup_txt, rl_txt])
    acoes = ' '.join([acoes_map[p] for p in probs if p in acoes_map])
    return interp, acoes

quadro[['interpretacao_operacional', 'acoes_recomendadas']] = quadro.apply(
    lambda r: pd.Series(interpretar(r)), axis=1
)

print('Quadro com interpretacao (head 20):')
display(quadro.head(20))



Quadro com interpretacao (head 20):


,dominio,forno,n_files,completude,validade,acuracia_proxy,tempestividade,unicidade,consistencia,time_cols_detectadas,interpretacao_operacional,acoes_recomendadas
0,Consumo Fornos,F1,7,0.008234,0.824367,0.048059,NaN,0.188419,0.714286,Data,Risco moderado/alto para uso direto em modelag...,Investigar lacunas e padronizar preenchimento/...
1,Consumo Fornos,F2,8,0.008489,0.824302,0.042272,NaN,0.185917,0.625000,Data,Risco moderado/alto para uso direto em modelag...,Investigar lacunas e padronizar preenchimento/...
2,Consumo Fornos,F3,8,0.005964,0.823455,0.052146,NaN,0.179358,0.500000,Data,Risco moderado/alto para uso direto em modelag...,Investigar lacunas e padronizar preenchimento/...
3,Consumo Fornos,F4,8,0.011532,0.826249,0.062939,NaN,0.181300,0.625000,Data,Risco moderado/alto para uso direto em modelag...,Investigar lacunas e padronizar preenchimento/...
4,Consumo Fornos,F5,8,0.012073,0.825726,0.064279,NaN,0.181750,0.750000,Data,Risco moderado/alto para uso direto em modelag...,Investigar lacunas e padronizar preenchimento/...
5,Corridas,F1,8,0.992644,0.798625,0.000271,NaN,1.000000,1.000000,Data_Base,Risco moderado/alto para uso direto em modelag...,Padronizar parse/tipos/unidades e bloquear val...
6,Corridas,F2,8,0.979425,0.797562,0.000111,NaN,1.000000,0.625000,Data_Base,Risco moderado/alto para uso direto em modelag...,Investigar lacunas e padronizar preenchimento/...
7,Corridas,F3,8,0.995088,0.785075,0.000675,NaN,1.000000,1.000000,Data_Base,Risco moderado/alto para uso direto em modelag...,Padronizar parse/tipos/unidades e bloquear val...
8,Corridas,F4,8,0.992663,0.779750,0.000591,NaN,1.000000,1.000000,Data_Base,Risco moderado/alto para uso direto em modelag...,Padronizar parse/tipos/unidades e bloquear val...
9,Corridas,F5,8,0.992449,0.779704,0.000566,NaN,1.000000,1.000000,Data_Base,Risco moderado/alto para uso direto em modelag...,Padronizar parse/tipos/unidades e bloquear val...


In [7]:
out_csv = Path('/home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R3Q/quadro_qualidade_por_forno.csv')
out_md = Path('/home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R3Q/quadro_qualidade_por_forno.md')
out_rel = Path('/home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R3Q/Relatorio_Quadro_Qualidade_Entregavel_1A.md')
out_sum = Path('/home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R3Q/r3q_summary.json')

quadro.to_csv(out_csv, index=False)
out_md.write_text(quadro.to_markdown(index=False), encoding='utf-8')

intro = (
"# Quadro consolidado de qualidade de dados por forno e dominio\n\n"
"Este documento apresenta, por forno e por conjunto de variaveis (dominio), os indicadores de completude, validade, "
"acuracia (proxy), tempestividade, unicidade e consistencia. Os indicadores sao calculados diretamente a partir dos CSVs, "
"com leitura em chunks para lidar com grande volume. O indicador de acuracia e tratado como proxy (plausibilidade estatistica), "
"pois nao ha, nesta fase, referencia metrologica completa para afirmar acuracia fisica.\n\n"
)

final = (
"\n\n# Comentario final: conclusoes e decisoes para o cliente\n\n"
"O quadro indica onde os dados ja permitem analise e modelagem com risco baixo, e onde ha risco de interpretacao por lacunas, "
"inconsistencias de schema, ausencia de timestamp confiavel, duplicidades ou sinais com alta taxa de outliers (proxy). As decisoes "
"imediatas para a producao sao: (1) nomear a coluna temporal oficial por dominio e padronizar timestamp; (2) definir chaves naturais "
"para deduplicacao; (3) versionar mudancas de schema e significado; (4) priorizar variaveis criticas para proxy do eletrodo e definir "
"procedimentos de verificacao de acuracia (calibracao/checagens cruzadas).\n"
)

out_rel.write_text(intro + quadro.to_markdown(index=False) + final, encoding='utf-8')

summary = {
  'run_ts': RUN_TS,
  'n_rows_quadro': int(len(quadro)),
  'out_csv': str(out_csv),
  'out_relatorio': str(out_rel)
}
out_sum.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('Salvos:')
print('-', out_csv)
print('-', out_md)
print('-', out_rel)
print('-', out_sum)
print('Resumo:')
print(json.dumps(summary, ensure_ascii=False, indent=2))



Salvos:
- /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R3Q/quadro_qualidade_por_forno.csv
- /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R3Q/quadro_qualidade_por_forno.md
- /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R3Q/Relatorio_Quadro_Qualidade_Entregavel_1A.md
- /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R3Q/r3q_summary.json
Resumo:
{
  "run_ts": "2026-01-06_083804",
  "n_rows_quadro": 19,
  "out_csv": "/home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R3Q/quadro_qualidade_por_forno.csv",
  "out_relatorio": "/home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R3Q/Relatorio_Quadro_Qualidade_Entregavel_1A.md"
}
